In [71]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import  (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [72]:
phones_df = pd.read_csv('datasets/transformed_data/phones.csv')

In [73]:
# categorical_features_df = pd.read_csv('datasets/transformed_data/categorical_features.csv')
# encoder = joblib.load("models/features_encoder.pkl")
# categorical_features
# encoder.dtype
# categorical_features = categorical_features_df['features'].to_list()
# decoded = encoder.inverse_transform(phones_df[categorical_features])
# phones_df[categorical_features] = decoded
# phones_df.shape

In [74]:
X_data = phones_df.drop(columns=['price'])
y_data = phones_df['price']

X, X_test, y, y_test = train_test_split(
    X_data,
    y_data,
    test_size=.1
)

In [75]:
POP_SIZE = 10
N_FEATURES = len(X_data.columns)

In [76]:
population = []

for _ in range(POP_SIZE):
    feature_mask = np.random.randint(
        0,
        2,
        N_FEATURES
    )
    
    if feature_mask.sum() == 0:
        feature_mask[np.random.randint(N_FEATURES)] = 1
    
    chromosome = np.concatenate([
        feature_mask,
        [
            np.random.randint(50, 501),
            np.random.randint(3, 31),
            np.random.randint(2, 16),
            np.random.randint(1, 11),
            np.random.randint(0, 3)
        ]]
    )
    
    population.append(chromosome)
    
population = np.array(population,dtype=object)

population[0]
    

array([0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1,
       1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 226, 20, 14, 1, 1], dtype=object)

In [77]:
def decode(chromosome):
    
    feature_mask = chromosome[:N_FEATURES].astype(bool)
    
    n_estimators = int(chromosome[N_FEATURES])    
    max_depth = int(chromosome[N_FEATURES + 1])
    min_split = int(chromosome[N_FEATURES + 2])
    min_leaf = int(chromosome[N_FEATURES + 3])
    max_features_gene = int(chromosome[N_FEATURES + 4])
    
    max_features_map = {
        0:"sqrt",
        1:"log2",
        2: None
    }
    
    params = {
        "n_estimators" : n_estimators,
        "max_depth" : max_depth,
        "min_samples_split": min_split,
        "min_samples_leaf": min_leaf,
        "max_features": max_features_map[max_features_gene]
    }
    
    
    return feature_mask, params 

In [78]:
mask, params = decode(population[0])

mask, params

(array([False, False, False,  True,  True,  True,  True,  True, False,
        False,  True, False, False, False, False,  True, False,  True,
        False,  True,  True,  True,  True, False, False,  True, False,
         True, False,  True, False, False, False, False, False, False,
        False, False, False, False, False, False,  True, False, False,
         True, False, False,  True, False, False]),
 {'n_estimators': 226,
  'max_depth': 20,
  'min_samples_split': 14,
  'min_samples_leaf': 1,
  'max_features': 'log2'})

In [79]:

from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
global_best_score = 0
global_best_chromosome = []

kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

def fitness_function(chromosome, X, y):
    feature_mask, params = decode(chromosome)
    
    if(feature_mask.sum()) == 0:
        return
    
    X_selected = X.loc[:, feature_mask]
    
    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )
    
    scores = cross_val_score(
        model,
        X_selected,
        y,
        cv=kf,
        scoring="r2"
    )
    
    return np.mean(scores)


In [80]:
fitness = np.array([
    fitness_function(chromosome, X, y )
    for chromosome in population
])

In [ ]:
best_history = []
mean_history = []

best_history.append(fitness.max())
mean_history.append(fitness.mean())

In [82]:
best_idx = np.argmax(fitness)
# fitness[best_idx]
# best_idx
# fitness.max()
if fitness[best_idx] > global_best_score:
    global_best_score = fitness[best_idx]
    global_best = population[best_idx].copy()
    
# fitness
population[best_idx]

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 477, 10, 3, 4, 2], dtype=object)

In [83]:
global_best

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 477, 10, 3, 4, 2], dtype=object)

In [ ]:
def tournament_selection(population, fitness, k=3):
    selected = []
    
    pop_size = len(population)
    
    for _ in range(pop_size):
        
        idx = np.random.choice(
            pop_size,
            size=k,
            replace=False
        )
        
        tournament_fitness = fitness[idx]
        
        winner = idx[np.argmax(tournament_fitness)]
        
        selected.append(population[winner].copy())
        
    return np.array(selected, dtype=object)


def roulette_selection(population, fitness):
    
    adjusted = fitness - fitness.min() + 1e-10
    
    probability = adjusted / adjusted.sum()
    
    selected_index = np.random.choice(
        len(population),
        size = len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )
    
    
def boltzmann_selection(population, fitness, temperature):
    
    scaled = np.exp(
        (fitness-np.max(fitness)) / temperature
    )
    
    probability = scaled / scaled.sum()
    
    selected_index = np.random.choice(
        len(population),
        size=len(population),
        replace=True,
        p=probability
    )
    
    return np.array(
        [population[i].copy() for i in selected_index],
        dtype=object
    )
    

In [ ]:
def crossover(parent1, parent2):
    child1 = parent1.copy()
    child2 = parent2.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    for i in range(N_FEATURES, len(parent1)):
        if np.random.rand() < 0.5:
            child1[i], child2[i] = child2[i], child1[i]
            
    return child1, child2

In [98]:
MUTATION_PROB = 0.2
SIGMA = 0.1

def gaussian_mutation(value,  lower, upper):
    std = SIGMA * (upper - lower)
    value += np.random.normal(0, std)
    value = np.clip(value, lower, upper)
    
    return int(round(value))



def mutate(chromosome):
    chromosome = chromosome.copy()
    
    for i in range(N_FEATURES):
        if np.random.rand() < MUTATION_PROB:
            chromosome[i] = 1 - chromosome[i]
            
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES ] = gaussian_mutation(
            chromosome[N_FEATURES ], 50, 500
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 1] = gaussian_mutation(
            chromosome[N_FEATURES + 1], 2, 50
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 2] = gaussian_mutation(
            chromosome[N_FEATURES + 2], 2, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 3] = gaussian_mutation(
            chromosome[N_FEATURES + 3], 1, 20
        )
    if np.random.rand() < MUTATION_PROB:
        chromosome[N_FEATURES + 4] = np.random.randint(3)
        
    return chromosome
    

In [ ]:

CROSSOVER_PROB = 0.8
parents = tournament_selection(population, fitness, k=3)
childs1 = []
childs2 = []
offspring = []

for i in range(0, POP_SIZE, 2):
    parent1 = parents[i]
    parent2 = parents[i+1]
    
    if np.random.rand() < CROSSOVER_PROB:
        child1, child2 = crossover(parent1, parent2)
    else:
        child1 = parent1.copy()
        child2 = parent2.copy()
    
    childs1.append(child1)
    childs2.append(child2)
    child1 = mutate(child1)
    child2 = mutate(child2)
    
    
    offspring.extend([child1, child2])
    # offspring.append([child1, child2])

new_population = np.array(offspring, dtype=object)

In [128]:
len(parents)

10

In [139]:
parents[0]

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 477, 10, 3, 4, 2], dtype=object)

In [140]:
parents[1]

array([0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 0, 469, 22, 7, 3, 1], dtype=object)

In [141]:
new_population[0]

array([0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 477, 22, 3, 5, 2], dtype=object)

In [90]:
# import numpy as np
# import matplotlib.pyplot as plt

# def fitness_function(x):
#     return x * np.sin(10 * np.pi * x) + 1

# POP_SIZE = 40
# GENERATIONS = 100
# X_MIN, X_MAX = -1.0, 2.0
# CROSSOVER_PROB = 0.9
# MUTATION_PROB = 0.2
# MUTATION_STD = 0.1

# np.random.seed(42)

# population = np.random.uniform(X_MIN, X_MAX, POP_SIZE)

# def tournament_selection(pop, fitness, k=3):
#     selected = []
#     for _ in range(len(pop)):
#         idx = np.random.choice(len(pop), k, replace=False)
#         selected.append(pop[idx[np.argmax(fitness[idx])]])
#     return np.array(selected)
        
# def arithmetic_crossover(p1, p2):
#     alpha = np.random.rand()
#     return alpha *  p1 + (1 - alpha) * p2, alpha * p2 + (1 - alpha)* p1

# def mutate(x):
#     if np.random.rand() < MUTATION_PROB:
#         x += np.random.normal(0, MUTATION_STD)
    
#     return np.clip(x, X_MIN, X_MAX)



# best_history = []
# mean_history = []

# for _ in range(GENERATIONS):
#     fitness = fitness_function(population)
#     best_history.append(np.max(fitness))
#     mean_history.append(np.mean(fitness))
    
#     parents = tournament_selection(population, fitness)
#     offspring = []
    
#     np.random.shuffle(parents)
#     for i in range(0, POP_SIZE, 2):
#         if np.random.rand() < CROSSOVER_PROB:
#             c1, c2 = arithmetic_crossover(parents[i], parents[i+1])
#         else:
#             c1, c2 = parents[i], parents[i + 1]
#         offspring.extend([mutate(c1), mutate(c2)])
#         x = np.linspace(X_MIN, X_MAX, 500)
#     population = np.array(offspring)

# plt.figure()
# plt.plot(best_history, label="Best Fitness")
# plt.plot(mean_history, label="Mean Fitness")
# plt.legend()
# plt.show()

# plt.figure()
# plt.plot(x, fitness_function(x))
# plt.scatter(population, fitness_function(population))
# plt.show()
    